In [3]:
!pip install xgboost

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder , LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)


In [5]:
df = pd.read_csv("/content/laptop_prices.csv")

In [6]:
df.head()

,Company,Product,TypeName,Inches,Ram,OS,Weight,Price_euros,Screen,ScreenW,...,RetinaDisplay,CPU_company,CPU_freq,CPU_model,PrimaryStorage,SecondaryStorage,PrimaryStorageType,SecondaryStorageType,GPU_company,GPU_model
0,Apple,MacBook Pro,Ultrabook,13.3,8,macOS,1.37,1339.69,Standard,2560,...,Yes,Intel,2.3,Core i5,128,0,SSD,No,Intel,Iris Plus Graphics 640
1,Apple,Macbook Air,Ultrabook,13.3,8,macOS,1.34,898.94,Standard,1440,...,No,Intel,1.8,Core i5,128,0,Flash Storage,No,Intel,HD Graphics 6000
2,HP,250 G6,Notebook,15.6,8,No OS,1.86,575.00,Full HD,1920,...,No,Intel,2.5,Core i5 7200U,256,0,SSD,No,Intel,HD Graphics 620
3,Apple,MacBook Pro,Ultrabook,15.4,16,macOS,1.83,2537.45,Standard,2880,...,Yes,Intel,2.7,Core i7,512,0,SSD,No,AMD,Radeon Pro 455
4,Apple,MacBook Pro,Ultrabook,13.3,8,macOS,1.37,1803.60,Standard,2560,...,Yes,Intel,3.1,Core i5,256,0,SSD,No,Intel,Iris Plus Graphics 650


In [7]:
df.shape

(1275, 23)

In [8]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
print(num_cols)
print(cat_cols)

['Inches', 'Ram', 'Weight', 'Price_euros', 'ScreenW', 'ScreenH', 'CPU_freq', 'PrimaryStorage', 'SecondaryStorage']
['Company', 'Product', 'TypeName', 'OS', 'Screen', 'Touchscreen', 'IPSpanel', 'RetinaDisplay', 'CPU_company', 'CPU_model', 'PrimaryStorageType', 'SecondaryStorageType', 'GPU_company', 'GPU_model']


In [9]:
print(df.isna().sum())

Company                 0
Product                 0
TypeName                0
Inches                  0
Ram                     0
OS                      0
Weight                  0
Price_euros             0
Screen                  0
ScreenW                 0
ScreenH                 0
Touchscreen             0
IPSpanel                0
RetinaDisplay           0
CPU_company             0
CPU_freq                0
CPU_model               0
PrimaryStorage          0
SecondaryStorage        0
PrimaryStorageType      0
SecondaryStorageType    0
GPU_company             0
GPU_model               0
dtype: int64


In [10]:
duplicate_mask = df.duplicated()
num_duplicates = duplicate_mask.sum()
print("Number of duplicate rows:", num_duplicates)

Number of duplicate rows: 0


In [11]:
for col in cat_cols:
  print(f"{col} : {df[col].nunique()}")

Company : 19
Product : 618
TypeName : 6
OS : 9
Screen : 4
Touchscreen : 2
IPSpanel : 2
RetinaDisplay : 2
CPU_company : 3
CPU_model : 93
PrimaryStorageType : 4
SecondaryStorageType : 4
GPU_company : 4
GPU_model : 110


In [12]:
df.drop(columns=["Product"], inplace=True)

In [13]:
df.tail()

,Company,TypeName,Inches,Ram,OS,Weight,Price_euros,Screen,ScreenW,ScreenH,...,RetinaDisplay,CPU_company,CPU_freq,CPU_model,PrimaryStorage,SecondaryStorage,PrimaryStorageType,SecondaryStorageType,GPU_company,GPU_model
1270,Lenovo,2 in 1 Convertible,14.0,4,Windows 10,1.80,638.0,Full HD,1920,1080,...,No,Intel,2.5,Core i7 6500U,128,0,SSD,No,Intel,HD Graphics 520
1271,Lenovo,2 in 1 Convertible,13.3,16,Windows 10,1.30,1499.0,Quad HD+,3200,1800,...,No,Intel,2.5,Core i7 6500U,512,0,SSD,No,Intel,HD Graphics 520
1272,Lenovo,Notebook,14.0,2,Windows 10,1.50,229.0,Standard,1366,768,...,No,Intel,1.6,Celeron Dual Core N3050,64,0,Flash Storage,No,Intel,HD Graphics
1273,HP,Notebook,15.6,6,Windows 10,2.19,764.0,Standard,1366,768,...,No,Intel,2.5,Core i7 6500U,1024,0,HDD,No,AMD,Radeon R5 M330
1274,Asus,Notebook,15.6,4,Windows 10,2.20,369.0,Standard,1366,768,...,No,Intel,1.6,Celeron Dual Core N3050,500,0,HDD,No,Intel,HD Graphics


In [14]:
RANDOM_STATE = 42

In [15]:
X = df.drop("Price_euros", axis=1)
Y = df["Price_euros"]

In [16]:
numerical_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
print(numerical_cols)
print(categorical_cols)

['Inches', 'Ram', 'Weight', 'ScreenW', 'ScreenH', 'CPU_freq', 'PrimaryStorage', 'SecondaryStorage']
['Company', 'TypeName', 'OS', 'Screen', 'Touchscreen', 'IPSpanel', 'RetinaDisplay', 'CPU_company', 'CPU_model', 'PrimaryStorageType', 'SecondaryStorageType', 'GPU_company', 'GPU_model']


In [17]:
X_train , X_test , y_train , y_test = train_test_split(X, Y, test_size=0.2, random_state=RANDOM_STATE)

In [18]:
preprocess = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

In [19]:
baseline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LinearRegression())
    ]
)

In [20]:
baseline.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Inches', 'Ram', 'Weight',
                                                   'ScreenW', 'ScreenH',
                                                   'CPU_freq', 'PrimaryStorage',
                                                   'SecondaryStorage']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Company', 'TypeName', 'OS',
                                                   'Screen', 'Touchscreen',
                                                   'IPSpanel', 'RetinaDisplay',
                                                   'CPU_company', 'CPU_model',
                                                   'PrimaryStorageType',
                                                   'SecondaryStorageType',
                                                   'GPU_company',
                                                   'GPU_model'])])),
                ('model', LinearRegression())])

In [21]:
train_baseline_predict = baseline.predict(X_train)
test_baseline_predict = baseline.predict(X_test)

In [22]:
train_baseline_rmse = root_mean_squared_error(y_train, train_baseline_predict)
train_baseline_mae = mean_absolute_error(y_train, train_baseline_predict)
train_baseline_r2 = r2_score(y_train, train_baseline_predict)

print("\n=== TRAIN BASELINE METRICS (LinearRegression) ===")
print(f"RMSE: {train_baseline_rmse:.3f}")
print(f"MAE : {train_baseline_mae:.3f}")
print(f"R2  : {train_baseline_r2:.3f}")


=== TRAIN BASELINE METRICS (LinearRegression) ===
RMSE: 223.099
MAE : 155.583
R2  : 0.898


In [23]:
test_baseline_rmse = root_mean_squared_error(y_test, test_baseline_predict)
test_baseline_mae = mean_absolute_error(y_test, test_baseline_predict)
test_baseline_r2 = r2_score(y_test, test_baseline_predict)

print("\n=== TEST BASELINE METRICS (LinearRegression) ===")
print(f"RMSE: {test_baseline_rmse:.3f}")
print(f"MAE : {test_baseline_mae:.3f}")
print(f"R2  : {test_baseline_r2:.3f}")


=== TEST BASELINE METRICS (LinearRegression) ===
RMSE: 307.101
MAE : 218.772
R2  : 0.810


In [24]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [25]:
models = {
    "Linear Regression" : LinearRegression(),
    "Lasso" : Lasso(),
    "Ridge" : Ridge(),
    "DecisionTreeRegressor" : DecisionTreeRegressor(random_state=RANDOM_STATE),
    "RandomForestRegressor" : RandomForestRegressor(random_state=RANDOM_STATE),
    "GradientBoostingRegressor" : GradientBoostingRegressor(random_state=RANDOM_STATE),
    "XGBRegressor" : XGBRegressor(
        random_state=RANDOM_STATE,
        objective="reg:squarederror",
        eval_metric="rmse"
        )
    }

In [26]:
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

In [27]:
rows = []
for name , model in models.items():
    pipe = Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", model)
        ]
    )
    score = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({
        "model": name,
        "rmse": -score["test_rmse"].mean(),
        "mae": -score["test_mae"].mean(),
        "r2": score["test_r2"].mean()
    })

table = pd.DataFrame(rows).sort_values("rmse")
print(table)


                       model        rmse         mae        r2
6               XGBRegressor  292.327086  181.077657  0.821254
4      RandomForestRegressor  298.631912  181.991054  0.814033
2                      Ridge  301.970711  211.678646  0.809384
1                      Lasso  309.707148  216.157568  0.799752
5  GradientBoostingRegressor  312.098498  203.842579  0.795939
0          Linear Regression  319.824037  220.492247  0.785673
3      DecisionTreeRegressor  418.232299  244.802632  0.636731


In [28]:
table

,model,rmse,mae,r2
6,XGBRegressor,292.327086,181.077657,0.821254
4,RandomForestRegressor,298.631912,181.991054,0.814033
2,Ridge,301.970711,211.678646,0.809384
1,Lasso,309.707148,216.157568,0.799752
5,GradientBoostingRegressor,312.098498,203.842579,0.795939
0,Linear Regression,319.824037,220.492247,0.785673
3,DecisionTreeRegressor,418.232299,244.802632,0.636731


In [29]:
best_row = table.sort_values("rmse").iloc[0]

best_model_name = best_row["model"]
best_rmse = best_row["rmse"]

print("Best model based on CV RMSE:")
print("Model :", best_model_name)
print("CV RMSE:", best_rmse)

Best model based on CV RMSE:
Model : XGBRegressor
CV RMSE: 292.3270857768831


In [30]:
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [3, 5, 7],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
    "model__min_child_weight": [1, 3],
    "model__gamma": [0, 0.1]
}

In [31]:
xgb_pipe = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", XGBRegressor(random_state=RANDOM_STATE))
    ]
)

In [32]:
grid = GridSearchCV(xgb_pipe, param_grid, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)

In [33]:
grid.fit(X_train, y_train)

GridSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['Inches',
                                                                          'Ram',
                                                                          'Weight',
                                                                          'ScreenW',
                                                                          'ScreenH',
                                                                          'CPU_freq',
                                                                          'PrimaryStorage',
                                                                          'SecondaryStorage']),
                                                                        ('cat',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['Company',
                                                                          'TypeName',
                                                                          'OS',
                                                                          'Screen',
                                                                          'Touchscreen',
                                                                          '...
                                                     multi_strategy=None,
                                                     n_estimators=None,
                                                     n_jobs=None,
                                                     num_parallel_tree=None, ...))]),
             n_jobs=-1,
             param_grid={'model__colsample_bytree': [0.8, 1.0],
                         'model__gamma': [0, 0.1],
                         'model__learning_rate': [0.01, 0.05, 0.1],
                         'model__max_depth': [3, 5, 7],
                         'model__min_child_weight': [1, 3],
                         'model__n_estimators': [100, 200, 300],
                         'model__subsample': [0.8, 1.0]},
             scoring='neg_root_mean_squared_error')

In [34]:
print("\n=== XGB ===")
print("Best CV RMSE:", -grid.best_score_)
print("Best params:", grid.best_params_)


=== XGB ===
Best CV RMSE: 280.61823850749454
Best params: {'model__colsample_bytree': 1.0, 'model__gamma': 0.1, 'model__learning_rate': 0.05, 'model__max_depth': 7, 'model__min_child_weight': 1, 'model__n_estimators': 300, 'model__subsample': 0.8}


In [35]:
xgb_best = Pipeline(
    steps = [
        ("preprocess", preprocess),
        ("model",  XGBRegressor(
                learning_rate=0.05,
                gamma=0.1,
                max_depth=7,
                min_child_weight=1,
                n_estimators=300,
                subsample=0.8,
                colsample_bytree=1.0)
        )
    ]
)

In [36]:
xgb_best.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Inches', 'Ram', 'Weight',
                                                   'ScreenW', 'ScreenH',
                                                   'CPU_freq', 'PrimaryStorage',
                                                   'SecondaryStorage']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Company', 'TypeName', 'OS',
                                                   'Screen', 'Touchscreen',
                                                   'IPSpanel', 'RetinaDisplay',
                                                   'CPU_company', 'CPU_model',
                                                   'PrimaryStorageType',...
                              feature_types=None, feature_weights=None,
                              gamma=0.1, grow_policy=None, importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=7, max_leaves=None, min_child_weight=1,
                              missing=nan, monotone_constraints=None,
                              multi_strategy=None, n_estimators=300,
                              n_jobs=None, num_parallel_tree=None, ...))])

In [37]:
train_final_pred = xgb_best.predict(X_train)

train_final_rmse = root_mean_squared_error(y_train, train_final_pred)
train_final_mae = mean_absolute_error(y_train, train_final_pred)
train_final_r2 = r2_score(y_train, train_final_pred)

print("\n=== FINAL MODEL (Tuned XGB) Train Performance ===")
print(f"RMSE: {train_final_rmse:.3f}")
print(f"MAE : {train_final_mae:.3f}")
print(f"R2  : {train_final_r2:.3f}")


=== FINAL MODEL (Tuned XGB) Train Performance ===
RMSE: 71.037
MAE : 51.585
R2  : 0.990


In [38]:
test_final_pred = xgb_best.predict(X_test)

test_final_rmse = root_mean_squared_error(y_test, test_final_pred)
test_final_mae = mean_absolute_error(y_test, test_final_pred)
test_final_r2 = r2_score(y_test, test_final_pred)

print("\n=== FINAL MODEL (Tuned XGB) Test Performance ===")
print(f"RMSE: {test_final_rmse:.3f}")
print(f"MAE : {test_final_mae:.3f}")
print(f"R2  : {test_final_r2:.3f}")


=== FINAL MODEL (Tuned XGB) Test Performance ===
RMSE: 238.442
MAE : 164.095
R2  : 0.885


In [39]:
def predict_laptop_price(
    model,
    company: str,
    product: str,
    typename: str,
    inches: float,
    ram: int,
    os: str,
    weight: float,
    screen: str,
    screenw: int,
    screenh: int,
    touchscreen: str,
    ipspanel: str,
    retinadisplay: str,
    cpu_company: str,
    cpu_freq: float,
    cpu_model: str,
    primarystorage: int,
    secondarystorage: int,
    primarystoragetype: str,
    secondarystoragetype: str,
    gpu_company: str,
    gpu_model: str
) -> float:
    """
    Predict laptop price for one laptop.
    """

    new_row = pd.DataFrame([{
        "Company": company,
        "Product": product,
        "TypeName": typename,
        "Inches": inches,
        "Ram": ram,
        "OS": os,
        "Weight": weight,
        "Screen": screen,
        "ScreenW": screenw,
        "ScreenH": screenh,
        "Touchscreen": touchscreen,
        "IPSpanel": ipspanel,
        "RetinaDisplay": retinadisplay,
        "CPU_company": cpu_company,
        "CPU_freq": cpu_freq,
        "CPU_model": cpu_model,
        "PrimaryStorage": primarystorage,
        "SecondaryStorage": secondarystorage,
        "PrimaryStorageType": primarystoragetype,
        "SecondaryStorageType": secondarystoragetype,
        "GPU_company": gpu_company,
        "GPU_model": gpu_model
    }])

    return float(xgb_best.predict(new_row)[0])

In [40]:
example_pred = predict_laptop_price(
    model=xgb_best,
    company="Dell",
    product="Inspiron 15",
    typename="Notebook",
    inches=15.6,
    ram=8,
    os="Windows 10",
    weight=2.1,
    screen="Full HD",
    screenw=1920,
    screenh=1080,
    touchscreen="No",
    ipspanel="Yes",
    retinadisplay="No",
    cpu_company="Intel",
    cpu_freq=2.5,
    cpu_model="Core i5 7200U",
    primarystorage=256,
    secondarystorage=0,
    primarystoragetype="SSD",
    secondarystoragetype="No",
    gpu_company="Intel",
    gpu_model="HD Graphics 620"
)

print(f"Predicted Laptop Price: €{example_pred:.2f}")

Predicted Laptop Price: €970.29


In [41]:
import pickle

In [43]:
import pickle

with open("laptop_price_model.sav", "wb") as file:
    pickle.dump(xgb_best, file)

print("Model saved successfully!")

Model saved successfully!


In [44]:
with open("laptop_price_model.sav", "rb") as file:
    loaded_model = pickle.load(file)

print("Loaded successfully!")

Loaded successfully!


In [45]:
import os
print(os.path.getsize("laptop_price_model.sav"))

1045691
